# Multi-Session LeJEPA Experiment Runner

Train and evaluate the LeJEPA teacher model on multi-session neural spike data.

**Order of operations:**
1. Set `CONFIG_PATH` in the Config cell.
2. Run all cells top to bottom.
3. Results (CSV + plots) are saved under `results_out_path` from your config.
4. Training curves, test metrics, and latent-space UMAP also render inline below.

**Prerequisite:** Run the dataset pipeline first (`experiments/data/create_dataset.py`) so a Parquet exists at your config's `data_path`.

Clone Repo

In [ ]:
!git clone https://github.com/jacobposchl/jepsyn
!pip install torch_brain==0.1.0
!pip install -q snntorch temporaldata umap-learn pyarrow pyyaml

In [ ]:
import os
import sys
from pathlib import Path

# The repo is cloned to /content/jepsyn in Colab.
REPO_ROOT = Path("/content/jepsyn")
os.chdir(REPO_ROOT)
print(f"CWD: {Path.cwd()}")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "experiments" / "multi_session"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from multi_session import load_checkpoint, train_lejepa, train_mae
from jepsyn.utils import (evaluate_model, identify_units, load_and_prepare_data,
                           run_linear_probe, save_results, verify_config)

print("Imports OK")

## Configuration

Point `CONFIG_PATH` at your experiment YAML. See `configs/lejepa_lif_visual_cortex.yaml` for a full template with all encoder, predictor, and training fields.

At minimum, `results_out_path` must be set in the config for checkpoints and plots to be saved to disk.

In [ ]:
# Point CONFIG_PATH at your experiment YAML to select the model.
# e.g. lejepa_visual_cortex.yaml (SIGReg), vicreg_visual_cortex.yaml (VICReg)
CONFIG_PATH = Path("experiments/multi_session/configs/lejepa_visual_cortex.yaml")

config = verify_config(CONFIG_PATH)

tcfg = config["training_config"]
mcfg = config["model_config"]
reg_type = tcfg.get("reg_type", "sigreg").lower()

print(f"Config loaded:      {CONFIG_PATH}")
print(f"data_path:          {config['data_path']}")
print(f"results_out_path:   {config.get('results_out_path', '(not set — results will not be saved to disk)')}")
print()
print(f"reg_type:           {reg_type}")
print(f"epochs:             {tcfg.get('epochs', 100)}")
print(f"batch_size:         {tcfg.get('batch_size', 32)}")
print(f"lr:                 {tcfg.get('lr', 1e-4)}")
print(f"mask_ratio:         {tcfg.get('mask_ratio', 0.5)}")
print(f"ema_momentum:       {tcfg.get('ema_momentum', 0.996)}")
if reg_type == "vicreg":
    print(f"vic_sim:            {tcfg.get('vic_sim', 25.0)}")
    print(f"vic_std:            {tcfg.get('vic_std', 25.0)}")
    print(f"vic_cov:            {tcfg.get('vic_cov', 1.0)}")
else:
    print(f"lambd (SIGReg):     {tcfg.get('lambd', 0.05)}")
    print(f"num_slices:         {tcfg.get('num_slices', 256)}")
print(f"unit_dropout:       {tcfg.get('unit_dropout', 0.0)}")
print(f"unit_id_steps:      {tcfg.get('unit_id_steps', 200)}")
print(f"unit_id_lr:         {tcfg.get('unit_id_lr', 1e-3)}")
print()
print(f"d_model:            {mcfg.get('d_model', 256)}")
print(f"n_latents:          {mcfg.get('n_latents', 64)}")
print(f"window_size_s:      {mcfg.get('window_size_s', 0.4)}")
print(f"rope_t_min:         {mcfg.get('rope_t_min', 1e-3)}")
print(f"rope_t_max:         {mcfg.get('rope_t_max', 4.0)}")
print(f"use_delimiter_tokens: {mcfg.get('use_delimiter_tokens', True)}")

## Data Loading

Loads the Parquet from `data_path`, validates schema and integrity, then splits windows by `session_id` into train / val / test sets.

In [ ]:
train_loader, val_loader, test_loader, unit_maps, test_session_ids = load_and_prepare_data(config)

print(f"\nSessions in unit_maps: {len(unit_maps)}")
print(f"Units per session:     min={min(len(m) for m in unit_maps.values())}, "
      f"max={max(len(m) for m in unit_maps.values())}")
print(f"Train batches:         {len(train_loader)}")
print(f"Val batches:           {len(val_loader)}")
print(f"Test batches:          {len(test_loader)}")
print(f"Test session IDs:      {test_session_ids}")

## Training

Trains the **context encoder** (online, gradients flow), **target encoder** (EMA copy, no gradients), and **predictor** (narrow Transformer).

Regularization is selected by `reg_type` in your config:
- **SIGReg** (default): `total = (1 - λ) * MSE(h_pred, h_tgt) + λ * SIGReg(h_ctx, h_tgt)`
- **VICReg**: `total = vic_sim * MSE(h_pred, h_tgt) + vic_std * var_loss + vic_cov * cov_loss`

After training, the checkpoint is saved to `<results_out_path>/lejepa_checkpoint.pt`.

In [ ]:
jepa_model, jepa_train_metrics = train_lejepa(config, train_loader, val_loader, unit_maps)
print("\nTraining complete.")
jepa_train_metrics.tail()

## Training Results

Saves `metrics.csv` and `training_curves.png` to `<results_out_path>/LeJEPA/training/`, then renders the curves inline.

In [ ]:
save_results(stage="LeJEPA", phase="training", metrics=jepa_train_metrics, config=config)

!zip -r LeJEPA_export.zip results
from google.colab import files
files.download('LeJEPA_export.zip')

In [ ]:
# Inline training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("LeJEPA - Training Curves")

axes[0].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_loss"], label="train")
axes[0].plot(jepa_train_metrics["epoch"], jepa_train_metrics["val_loss"], label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Total Loss")
axes[0].legend()

axes[1].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_pred_loss"], label="pred loss")
axes[1].plot(jepa_train_metrics["epoch"], jepa_train_metrics["train_reg_loss"], label="reg loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Pred vs Reg Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Full per-epoch metrics table
jepa_train_metrics

## Evaluation — Standalone (works after restart)

Re-run the cells below any time, including after a runtime restart, as long as `CONFIG_PATH` is set and a checkpoint exists at `<results_out_path>/lejepa_checkpoint.pt`.

Steps:
1. Reload data and checkpoint
2. Adapt test-session unit embeddings (`identify_units`)
3. Evaluate on test set (`evaluate_model`)
4. Run downstream linear probes (`run_linear_probe`)

In [ ]:
# Reload config and data. train/val loaders are discarded; only the test split is used below.
config = verify_config(CONFIG_PATH)
_, _, test_loader, unit_maps, test_session_ids = load_and_prepare_data(config)

# Derive stage label from reg_type (mirrors multi_session.py main())
reg_type = config.get("training_config", {}).get("reg_type", "sigreg").lower()
if reg_type == "vicreg":
    stage_name = "VICReg"
elif reg_type == "no_reg":
    stage_name = "LeJEPA-NoReg"
else:
    stage_name = "LeJEPA"

mask_ratio = config.get("training_config", {}).get("mask_ratio", 0.5)

# Load checkpoint — unit_maps are embedded in the checkpoint file.
ckpt_path = Path(config["results_out_path"]) / "lejepa_checkpoint.pt"
model, _, _ = load_checkpoint(ckpt_path)

print(f"\nStage:         {stage_name}")
print(f"Checkpoint:    {ckpt_path}")
print(f"Test sessions: {test_session_ids}")

### Unit Identification

Adapts test-session unit and session embedding tables using the JEPA self-supervised objective with all shared weights frozen. No test labels are used — only the embedding rows for test sessions are trainable.

In [ ]:
identify_units(model, test_loader, test_session_ids, config)

### Test Metrics

In [ ]:
test_metrics = evaluate_model(model, test_loader, stage=stage_name, mask_ratio=mask_ratio)
save_results(stage=stage_name, phase="test", metrics=test_metrics, config=config)

# Display saved plots inline
results_path = config.get("results_out_path")
if results_path:
    for fname in ["test_metrics.png", "latent_space.png", "latent_space_change.png"]:
        p = Path(results_path) / stage_name / "test" / fname
        if p.exists():
            print(fname)
            display(Image(filename=str(p)))

### Linear Probes

Fits 5-fold cross-validated linear classifiers on frozen test-session representations:
- **`is_change`** — binary: passive vs. active stimulus block
- **`image_name`** — multi-class: image identity
- **`session_identity`** — multi-class: which session a window came from

In [ ]:
probe_results = run_linear_probe(model, test_loader, stage=stage_name)

# Convert dict → DataFrame (one row per probe task)
probe_df = pd.DataFrame(probe_results).T

results_path = config.get("results_out_path")
if results_path:
    probe_csv = Path(results_path) / stage_name / "test" / "probe_results.csv"
    probe_df.to_csv(probe_csv)
    print(f"Saved probe results to {probe_csv}")

probe_df

### Download Results

In [ ]:
!zip -r LeJEPA_export.zip results
from google.colab import files
files.download('LeJEPA_export.zip')